In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [44]:
df = pd.read_csv("minesafe_100_rows_10_features.csv")

print(df.shape)
print(df["priority"].value_counts())

df.head()

(100, 11)
priority
HIGH        41
MEDIUM      35
LOW         13
CRITICAL    11
Name: count, dtype: int64


,incident_type,severity,workers_at_risk,injured_count,hazard_intensity,workers_in_danger_zone,nearest_worker_distance_m,communication_status,rescue_response_time_min,time_since_incident_min,priority
0,Water Increase,Moderate,11,2,25,10,34.5,Degraded,25.6,9,MEDIUM
1,Gas Leak,Moderate,14,4,35,4,8.5,Normal,23.3,33,MEDIUM
2,Electrical Fault,Serious,30,1,19,17,46.4,Normal,17.5,5,HIGH
3,Conveyor Accident,Critical,20,3,8,18,68.0,Normal,27.7,151,CRITICAL
4,Electrical Fault,Moderate,20,5,3,6,65.5,Normal,7.8,163,MEDIUM


In [45]:
FEATURES = [
    "incident_type",
    "severity",
    "workers_at_risk",
    "injured_count",
    "hazard_intensity",
    "workers_in_danger_zone",
    "nearest_worker_distance_m",
    "communication_status",
    "rescue_response_time_min",
    "time_since_incident_min"
]

TARGET = "priority"

X = df[FEATURES]
y = df[TARGET]

In [46]:
categorical_features = [
    "incident_type",
    "severity",
    "communication_status"
]

numeric_features = [
    "workers_at_risk",
    "injured_count",
    "hazard_intensity",
    "workers_in_danger_zone",
    "nearest_worker_distance_m",
    "rescue_response_time_min",
    "time_since_incident_min"
]

In [47]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [48]:
svc = SVC(
    probability=True,
    class_weight="balanced",
    random_state=42
)

In [49]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", svc)
])

In [50]:
param_grid = {
    "model__C": [0.1, 1, 10, 50, 100],
    "model__gamma": ["scale", "auto", 0.01, 0.1, 1],
    "model__kernel": ["rbf"]
}

In [51]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

In [52]:
grid_search.fit(X, y)

c:\Users\sayooj ramkumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto', ...], 'model__kernel': ['rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"v

In [53]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation accuracy:")
print(
    round(grid_search.best_score_ * 100, 2),
    "%"
)

Best parameters:
{'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}

Best cross-validation accuracy:
68.0 %


In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [55]:
best_model = grid_search.best_estimator_

best_model.fit(X_train, y_train)

predictions = best_model.predict(X_test)

c:\Users\sayooj ramkumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [56]:
accuracy = accuracy_score(y_test, predictions)

print(
    "Test Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

Test Accuracy: 50.0 %

Classification Report:
              precision    recall  f1-score   support

    CRITICAL       0.33      0.50      0.40         2
        HIGH       0.62      0.62      0.62         8
         LOW       1.00      0.33      0.50         3
      MEDIUM       0.38      0.43      0.40         7

    accuracy                           0.50        20
   macro avg       0.58      0.47      0.48        20
weighted avg       0.56      0.50      0.51        20



In [57]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)


Confusion Matrix:
[[1 0 0 1]
 [1 5 0 2]
 [0 0 1 2]
 [1 3 0 3]]
